# Mortgage Credit Risk — V2 Pipeline Runbook

This notebook is the documented entry point for the **V2 behavioural model**. It validates the project context, summarises the active configuration, runs the configured pipeline, and points to the resulting artifacts.

V3 is maintained separately and is intentionally out of scope here.

## Before you run

1. Activate the project environment and install the package/dependencies.
2. Ensure Java and PySpark are available when using the configured PySpark engine. Do not hard-code machine-specific Java or Hadoop paths in this notebook.
3. Review configuration files before running. The checked-in configuration skips ingestion and preprocessing, so compatible model-input data must already exist.
4. Run the cells from top to bottom. The pipeline writes artifacts below the configured data root.

## V2 scope

V2 creates one record per loan and observation age. The active configuration evaluates loan ages 6 and 12 and predicts future serious delinquency over 12 months.

Features may use origination data and performance history through the observation month only. Future performance constructs the target and must not enter the predictors.

In [4]:
from pathlib import Path
import os

# Supports launching from either the repository root or notebooks/.
project_path = Path.cwd().resolve()
if not (project_path / "config").is_dir():
    project_path = project_path.parent

if not (project_path / "config").is_dir():
    raise FileNotFoundError(
        "Could not find the project root. Start Jupyter from the repository root or notebooks/.",
    )

os.chdir(project_path)
print(f"Project root: {project_path}")

Project root: C:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk


In [5]:
from credit_risk.utils.config import read_config

config = read_config(project_path)
parameters = config["parameters"]
modelling = parameters["modelling"]
behavioral = parameters["behavioral"]
evaluation = parameters["evaluation"]

summary = {
    "approach": parameters["modelling_approach"],
    "engine": parameters["engine"],
    "observation_ages": behavioral["observation_ages"],
    "prediction_horizon_months": behavioral["prediction_horizon_months"],
    "target": parameters["target"]["name"],
    "algorithm": modelling["algorithm"],
    "train_vintages": modelling["vintages_train"],
    "validation_vintages": modelling["vintages_test"],
    "oot_vintages": modelling["vintages_oot"],
    "preprocessing_skipped": parameters["data"]["preprocess"]["skip"],
    "evaluation_mode": evaluation["mode"],
}

for name, value in summary.items():
    print(f"{name}: {value}")

approach: behavioral
engine: pyspark
observation_ages: [6, 12]
prediction_horizon_months: 12
target: future_90dpd_12m
algorithm: xgboost
train_vintages: [2015, 2016, 2017, 2018]
validation_vintages: [2019, 2020]
oot_vintages: [2021, 2022]
preprocessing_skipped: True
evaluation_mode: same_run


## Run the pipeline

The next cell runs every enabled stage. Confirm the configuration summary above before executing it. If preprocessing is enabled, ensure the raw data paths and Spark prerequisites are available.

In [6]:
from credit_risk import run_pipeline

run_pipeline(project_path)

11:16:37  INFO      utils.spark   Creating Spark session: master=local[16] driver_memory=14g shuffle_partitions=256


11:16:37  INFO      utils.spark   Spark session created: version=4.2.0 default_parallelism=256
11:16:37  INFO      pipeline      ━━ Pipeline started ━━ project=C:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk
11:16:37  INFO      ingest        Ingestion skipped by configuration
11:16:37  INFO      data_preprocess  Preprocessing skipped by configuration
11:16:37  INFO      reporting     Data-quality reporting skipped by configuration
11:16:37  INFO      modelling     Modelling pipeline skipped by configuration
11:16:40  INFO      modelling.artifacts_spark  Spark artifact loaded: path=data\05_artifacts\behavioral\test_unbalanced_quarterly\xgboost\preprocessor.joblib duration_seconds=3.35
11:16:40  INFO      modelling.artifacts_spark  Spark artifact loaded: path=data\05_artifacts\behavioral\test_unbalanced_quarterly\xgboost\model.joblib duration_seconds=0.12
11:16:40  INFO      evaluation    Evaluation model loaded: engine=pyspark mode=same_run version=test_unbalanced_quarterl

## Review artifacts

The model, preprocessor, training configuration, metrics, charts, threshold summary, and SHAP outputs are written below the configured artifacts directory. Review validation and OOT results together; a strong ranking metric alone does not establish calibration, stability, or suitability for credit decisioning.

In [7]:
artifact_root = (
    project_path
    / config["catalog"]["base"]
    / config["catalog"]["model"]["folder_name"]
    / modelling["version"]
    / modelling["algorithm"]
)

if artifact_root.exists():
    print(f"Artifacts: {artifact_root}")
    for path in sorted(artifact_root.iterdir()):
        print(f"- {path.name}")
else:
    print("No artifacts found yet. Run the pipeline or verify the configured version and algorithm.")

No artifacts found yet. Run the pipeline or verify the configured version and algorithm.


## Troubleshooting

- **Project root not found:** launch Jupyter from the repository root or notebooks/.
- **Spark/Java failure:** configure Java and PySpark in your environment, then restart the kernel.
- **Missing model inputs:** either enable ingestion/preprocessing with valid raw data paths or restore compatible persisted inputs.
- **Unexpected results:** record the configuration, artifact version, source vintages, and evaluation population before investigating metrics.